# ANALISIS ESTADISTICO DE DATASET : TELCO CHURN
## FUENTE : [KAGGLE](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

# AUTOR : César Mayta

# PASO 1 - IMPORTAMOS LIBRERIAS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno

# PASO 2 - EJECUTAMOS PANDAS MISSING EXTENSION NOTEBOOK

In [2]:
%run '/content/pandas_missing_extension.ipynb'

# PASO 3 - CARGAMOS DATASET

In [3]:
data_df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')
data_df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# PASO 4 - EDA

In [4]:
data_df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [5]:
data_df.dtypes.value_counts()

,count
object,18
int64,2
float64,1


In [6]:
data_df.shape

(7043, 21)

## ELIMINAMOS EL CUSTOMERID PORQUE NO ES REVELANTE PARA EL ANALISIS DE MI DATA

In [7]:
data_df.drop('customerID',axis=1,inplace=True)
# AXIS = 1 : INDICA QUE ES EN LAS COLUMNAS
# INPLACE=TRUE : ACTUALIZA EL MISMO DATAFRAME

# PASO 5 - TRATAMIENTO DE DATOS FALTANTES

## CUANTOS FILAS NULAS TENGO

In [8]:
data_df.missing.number_missing()

np.int64(0)

## IMPRIMIR COLUMNAS DEL DATASET

In [9]:
data_df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

## CAMBIAMOS LA NOMENCLATURA DE COLUMNAS A SNAKE_CASE

In [10]:
data_df_raw = data_df.copy()

In [11]:
def to_snake_case(name):
    import re
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('__([A-Z])', r'_\1', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)
    return name.lower()

data_df_raw.columns = [to_snake_case(col) for col in data_df_raw.columns]

In [12]:
data_df_raw.columns

Index(['gender', 'senior_citizen', 'partner', 'dependents', 'tenure',
       'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charges', 'total_charges', 'churn'],
      dtype='object')

## IDENTIFICAMOS LAS VARIABLES CATEGORICAS

In [13]:
data_df_clean = data_df_raw.copy()

In [14]:
data_df_clean.dtypes

,0
gender,object
senior_citizen,int64
partner,object
dependents,object
tenure,int64
phone_service,object
multiple_lines,object
internet_service,object
online_security,object
online_backup,object


In [15]:
# prompt: convierte la columna total_charges a float64

import numpy as np
# Replace empty strings with NaN and then convert to float64, coercing errors
data_df_clean['total_charges'] = data_df_clean['total_charges'].replace(' ', np.nan).astype(np.float64)

In [16]:
categorical_columns = data_df_clean.select_dtypes(object).columns
categorical_columns

Index(['gender', 'partner', 'dependents', 'phone_service', 'multiple_lines',
       'internet_service', 'online_security', 'online_backup',
       'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
       'contract', 'paperless_billing', 'payment_method', 'churn'],
      dtype='object')

# MOSTRAMOS LAS CATEGORICAS POR CADA VARIABLE, ES DECIR LOS POSIBLES VALORES QUE TIENE CADA VARIABLE CATEGORICA

In [17]:
for cc in categorical_columns:
  print("*"*50)
  print(data_df_clean[cc].value_counts())

**************************************************
gender
Male      3555
Female    3488
Name: count, dtype: int64
**************************************************
partner
No     3641
Yes    3402
Name: count, dtype: int64
**************************************************
dependents
No     4933
Yes    2110
Name: count, dtype: int64
**************************************************
phone_service
Yes    6361
No      682
Name: count, dtype: int64
**************************************************
multiple_lines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64
**************************************************
internet_service
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64
**************************************************
online_security
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64
**************************************************
o

## CREAMOS VARIABLES PARA LOS TIPOS DE CODIFICACION POR COLUMNAS
## ORDINAL_COLS  : SON LAS COLUMNAS QUE SERAN CODIFICADAS ORDINALMENTE
## ONEHOT_COLS : SON LAS COLUMNAS QUE SERAN CODIFICADAS CON ONE HOT ENCODING

In [18]:
ordinal_cols = ['gender','partner','dependents','phone_service','paperless_billing','churn']
onehot_cols = list(set(categorical_columns) - set(ordinal_cols))
onehot_cols

['payment_method',
 'streaming_tv',
 'device_protection',
 'online_security',
 'online_backup',
 'contract',
 'multiple_lines',
 'streaming_movies',
 'tech_support',
 'internet_service']

# CODIFICAMOS LAS VARIABLES CATEGORICAS USANDO UN TRANSFORMR DE SKLEARN

## CREAMO SUN TRANSFORMER

In [19]:
import sklearn.compose
import sklearn.preprocessing

In [20]:
transformer = sklearn.compose.make_column_transformer(
    (sklearn.preprocessing.OrdinalEncoder(),ordinal_cols),
    (sklearn.preprocessing.OneHotEncoder(),onehot_cols),
    remainder='passthrough'
)
transformer

ColumnTransformer(remainder='passthrough',
                  transformers=[('ordinalencoder', OrdinalEncoder(),
                                 ['gender', 'partner', 'dependents',
                                  'phone_service', 'paperless_billing',
                                  'churn']),
                                ('onehotencoder', OneHotEncoder(),
                                 ['payment_method', 'streaming_tv',
                                  'device_protection', 'online_security',
                                  'online_backup', 'contract', 'multiple_lines',
                                  'streaming_movies', 'tech_support',
                                  'internet_service'])])

## APLICAMOS EL TRANSFOMER A LA VARIABLES CATEGORICAS

In [21]:
data_transformed_df = (
    pd.DataFrame(
        transformer.fit_transform(data_df_clean),
        columns = transformer.get_feature_names_out(),
        index=data_df_clean.index
    )
)
data_transformed_df

,ordinalencoder__gender,ordinalencoder__partner,ordinalencoder__dependents,ordinalencoder__phone_service,ordinalencoder__paperless_billing,ordinalencoder__churn,onehotencoder__payment_method_Bank transfer (automatic),onehotencoder__payment_method_Credit card (automatic),onehotencoder__payment_method_Electronic check,onehotencoder__payment_method_Mailed check,...,onehotencoder__tech_support_No,onehotencoder__tech_support_No internet service,onehotencoder__tech_support_Yes,onehotencoder__internet_service_DSL,onehotencoder__internet_service_Fiber optic,onehotencoder__internet_service_No,remainder__senior_citizen,remainder__tenure,remainder__monthly_charges,remainder__total_charges
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,29.85,29.85
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,34.0,56.95,1889.50
2,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,53.85,108.15
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,45.0,42.30,1840.75
4,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,24.0,84.80,1990.50
7039,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,72.0,103.20,7362.90
7040,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,11.0,29.60,346.45
7041,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,4.0,74.40,306.60


# RENOMBRAMOS LAS COLUMNAS QUITANDOLES EL PREFIJO DE LA CODIFICACIÓN

In [22]:
data_transformed_df = data_transformed_df.rename(
    columns=lambda x: x.replace("ordinalencoder__", "").replace("onehotencoder__", "").replace("remainder__", "")
)

data_transformed_df

,gender,partner,dependents,phone_service,paperless_billing,churn,payment_method_Bank transfer (automatic),payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,...,tech_support_No,tech_support_No internet service,tech_support_Yes,internet_service_DSL,internet_service_Fiber optic,internet_service_No,senior_citizen,tenure,monthly_charges,total_charges
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,29.85,29.85
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,34.0,56.95,1889.50
2,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,53.85,108.15
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,45.0,42.30,1840.75
4,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,24.0,84.80,1990.50
7039,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,72.0,103.20,7362.90
7040,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,11.0,29.60,346.45
7041,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,4.0,74.40,306.60


# VOLVEMOS A DAR FORMATO SNAKE CASE A LAS COLUMNAS

In [23]:
data_transformed_df.columns = [to_snake_case(col) for col in data_transformed_df.columns]

In [24]:
data_transformed_df.columns

Index(['gender', 'partner', 'dependents', 'phone_service', 'paperless_billing',
       'churn', 'payment_method_bank transfer (automatic)',
       'payment_method_credit card (automatic)',
       'payment_method_electronic check', 'payment_method_mailed check',
       'streaming_tv_no', 'streaming_tv_no internet service',
       'streaming_tv_yes', 'device_protection_no',
       'device_protection_no internet service', 'device_protection_yes',
       'online_security_no', 'online_security_no internet service',
       'online_security_yes', 'online_backup_no',
       'online_backup_no internet service', 'online_backup_yes',
       'contract_month-to-month', 'contract_one year', 'contract_two year',
       'multiple_lines_no', 'multiple_lines_no phone service',
       'multiple_lines_yes', 'streaming_movies_no',
       'streaming_movies_no internet service', 'streaming_movies_yes',
       'tech_support_no', 'tech_support_no internet service',
       'tech_support_yes', 'internet_service_

## SIGO RENOMBRANDO LAS COLUMNAS PARA QUE SEAN MAS CORTAS

In [25]:
data_transformed_df = data_transformed_df.rename(
    columns=lambda x: x.replace("streaming_movies_", "movies_").replace("payment_method_", "payment_").replace("internet_service_", "internet_").replace("streaming_tv", "tv_").replace("online_security_", "security_").replace("online_backup_", "backup_").replace("no_internet_service", "none")
)
data_transformed_df

,gender,partner,dependents,phone_service,paperless_billing,churn,payment_bank transfer (automatic),payment_credit card (automatic),payment_electronic check,payment_mailed check,...,tech_support_no,tech_support_no internet service,tech_support_yes,internet_dsl,internet_fiber optic,internet_no,senior_citizen,tenure,monthly_charges,total_charges
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,29.85,29.85
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,34.0,56.95,1889.50
2,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,53.85,108.15
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,45.0,42.30,1840.75
4,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,24.0,84.80,1990.50
7039,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,72.0,103.20,7362.90
7040,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,11.0,29.60,346.45
7041,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,4.0,74.40,306.60


In [26]:
data_transformed_df.columns

Index(['gender', 'partner', 'dependents', 'phone_service', 'paperless_billing',
       'churn', 'payment_bank transfer (automatic)',
       'payment_credit card (automatic)', 'payment_electronic check',
       'payment_mailed check', 'tv__no', 'tv__no internet service', 'tv__yes',
       'device_protection_no', 'device_protection_no internet service',
       'device_protection_yes', 'security_no', 'security_no internet service',
       'security_yes', 'backup_no', 'backup_no internet service', 'backup_yes',
       'contract_month-to-month', 'contract_one year', 'contract_two year',
       'multiple_lines_no', 'multiple_lines_no phone service',
       'multiple_lines_yes', 'movies_no', 'movies_no internet service',
       'movies_yes', 'tech_support_no', 'tech_support_no internet service',
       'tech_support_yes', 'internet_dsl', 'internet_fiber optic',
       'internet_no', 'senior_citizen', 'tenure', 'monthly_charges',
       'total_charges'],
      dtype='object')

In [28]:
data_transformed_df = data_transformed_df.rename(
    columns={
        "payment_bank transfer (automatic)":"payment_transfer",
        "payment_credit card (automatic)":"payment_creditcard",
        "payment_electronic check":"payment_check",
        "payment_mailed check":"payment_mail",
        "tv__no":"tv_no",
        "tv__yes":"tv_yes",
        "tv__none":"tv_none",
        "device_protection_no internet service":"device_protection_noservice",
        "security_no internet service":"security_noservice",
        "backup_no internet service":"backup_noservice",
        "contract_one year":"contract_year",
        "contract_month-to-month":"contract_month",
        "contract_two year":"contract_twoyears",
        "multiple_lines_no phone service":"multiple_lines_noservice",
        "movies_no internet service":"movies_noservice",
        "tech_support_no internet service":"tech_support_noservice",
        "internet_fiber optic":"internet_fiber"
    }
)
data_transformed_df.columns

Index(['gender', 'partner', 'dependents', 'phone_service', 'paperless_billing',
       'churn', 'payment_transfer', 'payment_creditcard', 'payment_check',
       'payment_mail', 'tv_no', 'tv__no internet service', 'tv_yes',
       'device_protection_no', 'device_protection_noservice',
       'device_protection_yes', 'security_no', 'security_noservice',
       'security_yes', 'backup_no', 'backup_noservice', 'backup_yes',
       'contract_month', 'contract_year', 'contract_twoyears',
       'multiple_lines_no', 'multiple_lines_noservice', 'multiple_lines_yes',
       'movies_no', 'movies_noservice', 'movies_yes', 'tech_support_no',
       'tech_support_noservice', 'tech_support_yes', 'internet_dsl',
       'internet_fiber', 'internet_no', 'senior_citizen', 'tenure',
       'monthly_charges', 'total_charges'],
      dtype='object')

In [29]:
data_transformed_df

,gender,partner,dependents,phone_service,paperless_billing,churn,payment_transfer,payment_creditcard,payment_check,payment_mail,...,tech_support_no,tech_support_noservice,tech_support_yes,internet_dsl,internet_fiber,internet_no,senior_citizen,tenure,monthly_charges,total_charges
0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,29.85,29.85
1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,34.0,56.95,1889.50
2,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,53.85,108.15
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,45.0,42.30,1840.75
4,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,24.0,84.80,1990.50
7039,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,72.0,103.20,7362.90
7040,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,11.0,29.60,346.45
7041,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,4.0,74.40,306.60


In [30]:
data_transformed_df.dtypes

,0
gender,float64
partner,float64
dependents,float64
phone_service,float64
paperless_billing,float64
churn,float64
payment_transfer,float64
payment_creditcard,float64
payment_check,float64
payment_mail,float64


## UNA VEZ PROCESADO MI DATAFRAME GUARDO EL RESULTADO UN NUEVO CSV

In [31]:
data_transformed_df.to_csv('telco_data_transformed.csv',index=False,encoding='utf-8')